# Simulación de interacción SPARQL con datos privados en un POD de Solid

IMPORTANTE SEÑALAR: Este Jupyter no se conecta a un POD real, pero simula cómo una aplicación podría leer datos privados y combinarlos con los grafo RDF del proyecto MultiSolve.

Amnte todo, se cargan los datos del proyecto y del POD simulado

In [9]:
from rdflib import Graph

g = Graph()

# Grafo del proyecto
g.parse("../Datos/tarea1-multisolve-private.ttl", format="turtle")
g.parse("../Datos/tarea1-chile-geo-open.ttl", format="turtle")

# POD simulado
g.parse("Datos/solid_pod_simulado.ttl", format="turtle")

print("Triples cargados:", len(g))

Triples cargados: 175


En este ejemplo, el POD simulado contiene datos privados del cliente, tales como:

- Nombre
- Email
- Canal Preferido de Contacto
- Dirección
- Comuna
- Última Fecha de Servicio
- Preferencia de Servicio

Estos datos no deberían ser públicos en un sistema tradicional, no obstante, en Solid podrían ser gestionados bajo control del usuario

In [14]:
query_pod = """
PREFIX ex:  <http://example.org/pod#>

SELECT ?customerName ?contactEmail ?preferredContact ?lastServiceDate
WHERE {
  ?profile a ex:PrivateProfile ;
           ex:customerName ?customerName ;
           ex:contactEmail ?contactEmail ;
           ex:preferredContact ?preferredContact ;
           ex:lastServiceDate ?lastServiceDate .
}
ORDER BY ?customerName
"""

results_pod = list(g.query(query_pod))
for row in results_pod:
    print(row)

(rdflib.term.Literal('Cliente 1', lang='es'), rdflib.term.Literal('cliente1@example.org'), rdflib.term.Literal('WhatsApp', lang='es'), rdflib.term.Literal('2026-01-05', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#date')))
(rdflib.term.Literal('Cliente 2', lang='es'), rdflib.term.Literal('cliente2@example.org'), rdflib.term.Literal('Correo electrónico', lang='es'), rdflib.term.Literal('2026-01-07', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#date')))


Consulta combinada entre datos privados y abiertos

In [15]:
query_private_open = """
PREFIX ex:   <http://example.org/pod#>
PREFIX geo:  <http://example.org/chile-geo#>

SELECT ?customerName ?preferredContact ?street ?number ?comunaName
WHERE {
  ?profile a ex:PrivateProfile ;
           ex:customerName ?customerName ;
           ex:preferredContact ?preferredContact ;
           ex:street ?street ;
           ex:number ?number ;
           ex:locatedInComuna ?comuna .
  ?comuna geo:nombreComuna ?comunaName .
}
ORDER BY ?customerName
"""

results_private_open = list(g.query(query_private_open))
for row in results_private_open:
    print(row)

(rdflib.term.Literal('Cliente 1', lang='es'), rdflib.term.Literal('WhatsApp', lang='es'), rdflib.term.Literal('Av. Principal', lang='es'), rdflib.term.Literal('1234'), rdflib.term.Literal('Santiago', lang='es'))
(rdflib.term.Literal('Cliente 2', lang='es'), rdflib.term.Literal('Correo electrónico', lang='es'), rdflib.term.Literal('Pasaje Central', lang='es'), rdflib.term.Literal('456'), rdflib.term.Literal('Cerrillos', lang='es'))


onsulta combinada, POD simulado y órdenes de servicio

In [12]:
query_pod_orders = """
PREFIX ex:  <http://example.org/pod#>
PREFIX ms:  <http://example.org/multisolve#>

SELECT ?customerName ?orderId ?status
WHERE {
  ?profile a ex:PrivateProfile ;
           ex:customerName ?customerName .
  ?customer ms:customerName ?customerName .
  ?so a ms:ServiceOrder ;
      ms:orderId ?orderId ;
      ms:status ?status ;
      ms:hasCustomer ?customer .
}
ORDER BY ?customerName ?orderId
"""

results_pod_orders = list(g.query(query_pod_orders))
for row in results_pod_orders:
    print(row)

Ahora bien, ¿qué ocurriría si consultamos a un LLM de la misma forma que en la tarea 3?

In [16]:
def answer_from_pod_and_orders(customer_name, query_results):
    customer_rows = [r for r in query_results if r.customerName.toPython() == customer_name]
    if not customer_rows:
        return f"No se encontraron órdenes para {customer_name}."
    orders = ", ".join([f"{r.orderId.toPython()} ({r.status.toPython()})" for r in customer_rows])
    return f"{customer_name} tiene las siguientes órdenes registradas: {orders}."

print(answer_from_pod_and_orders("Cliente 1", results_pod_orders))
print(answer_from_pod_and_orders("Cliente 2", results_pod_orders))

No se encontraron órdenes para Cliente 1.
No se encontraron órdenes para Cliente 2.


Como se puede ver, este prototipo muestra la idea central del proyecto Solid:

- Los datos privados pueden estar separados del sistema central
- El acceso puede hacerse de manera controlada
- La aplicación puede combinar esos datos con información estructurada y datos abiertos para responder preguntas útiles

En un escenario real, esta interacción requeriría (claramente) autenticación y permisos sobre un POD real.  
Aquí se implementa una simulación conceptual, suficiente para mostrar la lógica del enfoque.

Ahora bien, ¿Qué ganaría MultiSolve con esto?

Aplicado al dominio del proyecto, este enfoque permitiría que el sistema pudiera:

- Consultar información privada del cliente solo con autorización
- Integrar esta información con órdenes de servicio
- Usar datos abiertos para enriquecer el contexto geográfico
- Ayudar a una mejor utilización de datos, respetando su privacidad